In [ ]:
import ee
import math
import geemap

# ── Authenticate & initialise the Earth Engine API ───────────────────────────
# This must be called before any ee.* objects are constructed.
# On first run it opens a browser auth flow; subsequent runs use cached creds.

ee.Authenticate()
ee.Initialize()

In [ ]:
# A blank map template should appear. If not, then the packages were not correctly instatlled

Map = geemap.Map()
Map

In [ ]:
# Input the AOI, and run this cell to view if your AOI is encompassed by the 1m DEM datset

# Load AOI
ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'

# Load AOI properly
AOI = ee.FeatureCollection(ASSET_ID)

# Load dataset
dataset = ee.ImageCollection('USGS/3DEP/1m')

# Visualization parameters
visualization = {
    'min': 0,
    'max': 3000,
    'palette': [
        '3ae237', 'b5e22e', 'd6e21f', 'fff705', 'ffd611', 'ffb613', 'ff8b13',
        'ff6e08', 'ff500d', 'ff0000', 'de0101', 'c21301', '0602ff', '235cb1',
        '307ef3', '269db1', '30c8e2', '32d3ef', '3be285', '3ff38f', '86e26f'
    ]
}

# Center map
Map.setCenter(-98.0, 40, 4)

# Add layers
Map.addLayer(dataset, visualization, 'AOI')

# Add your table (FeatureCollection) - make sure 'ASSET_ID' is defined
Map.addLayer(AOI, {'color': 'black'}, 'ASSET_ID')

# Display map
Map

In [ ]:
# CHM



# https://github.com/smorf-ntsg/naip-chm
# Review the GitHub page to learn about the dataset!

# =============================================================================
# 0.6 m CHM EXTRACTION (NAIP)
# =============================================================================

import ee
import geemap

# ═════════════════════════════════════════════════════════════════════════════
# 🎯 USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection stored in Earth Engine assets

Export_CRS = 'EPSG:5070'
# ^ Output coordinate reference system 

FOLDER = 'GEE_Exports_CHM'
# ^ Google Drive folder where GeoTIFF will be exported

FILE_PREFIX = 'MN_I-90_CHM'
# ^ Output filename

EXPORT_SCALE = 0.6
# ^ Native NAIP CHM resolution (~0.6 meters)

NODATA_VALUE = -9999
# ^ Standard NoData value for GIS compatibility

CHM_YEAR = 2023
# ^ Selects CHM dataset year. 2023 is the most recent

CHM_SCALE_FACTOR = 100.0
# ^ Dataset stores values as integers ×100 → convert back to meters
# ^ Leave as 100.0 and DO NOT MODIFY! 

# ═════════════════════════════════════════════════════════════════════════════
# LOAD INPUT DATA
# ═════════════════════════════════════════════════════════════════════════════

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Load AOI polygons

dissolved = export_poly.union().geometry()
# ^ Dissolve to single geometry (required for export region)

chm_collection = ee.ImageCollection(
    'projects/naip-chm/assets/conus-structure-model'
)
# ^ NAIP-derived canopy height model (CONUS coverage)

chm_mosaic = (
    chm_collection
    .filter(ee.Filter.eq('year', CHM_YEAR))
    # ^ Filter to selected year

    .filterBounds(export_poly)
    # ^ Only load tiles intersecting AOI

    .mosaic()
    # ^ Merge tiles into single seamless raster

    .divide(CHM_SCALE_FACTOR)
    # ^ Convert scaled integers → meters

    .rename('canopy_height_m')
    # ^ Output band name
)

# ═════════════════════════════════════════════════════════════════════════════
# PREPARE EXPORT IMAGE
# ═════════════════════════════════════════════════════════════════════════════

chm_export = (
    chm_mosaic
    .clip(export_poly)
    # ^ Restrict raster strictly to AOI

    .reproject(crs=Export_CRS, scale=EXPORT_SCALE)
    # ^ Force correct resolution + projection before export
)

ready_to_export = (
    chm_export
    .toFloat()
    # ^ Ensure Float32 output

    .unmask(NODATA_VALUE)
    # ^ Fill outside AOI / missing data with NoData value
)

# ═════════════════════════════════════════════════════════════════════════════
# INTERACTIVE MAP 
# ═════════════════════════════════════════════════════════════════════════════

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Center map on AOI

Map.add_basemap('SATELLITE')
# ^ Add imagery basemap for context

Map.addLayer(
    chm_mosaic.clip(export_poly),
    {
        'min': 0,
        'max': 40,
        'palette': [
            'ffffff',  # ground
            'c7e9b4',  # low vegetation
            '7fcdbb',
            '41b6c4',
            '2c7fb8',
            '253494'   # tall canopy
        ]
    },
    'CHM (m)'
)
# ^ Visualize canopy height

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ AOI boundary overlay

display(Map)

# ═════════════════════════════════════════════════════════════════════════════
# EXPORT 
# ═════════════════════════════════════════════════════════════════════════════

task = ee.batch.Export.image.toDrive(
    image=ready_to_export,
    # ^ Final prepared CHM raster

    description=FILE_PREFIX,
    # ^ Task name in GEE

    folder=FOLDER,
    # ^ Drive destination

    fileNamePrefix=FILE_PREFIX,
    # ^ Output filename

    region=dissolved,
    # ^ ✅ Use dissolved geometry (NOT FeatureCollection)

    scale=EXPORT_SCALE,
    # ^ Output resolution

    crs=Export_CRS,
    # ^ Output projection

    maxPixels=int(1e13)
    # ^ Prevents pixel limit errors
)

task.start()

# ═════════════════════════════════════════════════════════════════════════════
# STATUS
# ═════════════════════════════════════════════════════════════════════════════

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('CHM EXPORT STARTED')
print('AOI-based export (no partitioning)')
print('NoData value:', NODATA_VALUE)
print('CRS:', Export_CRS)
print('Scale:', EXPORT_SCALE, 'm')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

In [ ]:
# DEM 



# =============================================================================
# 1m DEM EXTRACTION 
# =============================================================================

import ee
import geemap

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining the corridor / study boundary.

Export_CRS = 'EPSG:5070'
# ^ Output coordinate reference system 

FOLDER = 'GEE_Exports_DEM'
# ^ Google Drive folder for exported GeoTIFF(s).

FILE_PREFIX = 'MN_I-90_DEM'
# ^ Output filename prefix.

EXPORT_SCALE = 1
# ^ 1 meter DEM resolution (native 3DEP resolution).

NODATA_VALUE = -9999
# ^ Standard GIS NoData value for ArcGIS / QGIS compatibility.

# =============================================================================
# LOAD DATA
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine assets.

dissolved = export_poly.union().geometry()
# ^ Dissolves AOI into a single geometry (required for export region).

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/1m')
    .filterBounds(export_poly)
    # ^ Keeps only DEM tiles intersecting AOI (performance optimization).

    .mosaic()
    # ^ Stitches tiles into one seamless raster.

    .select(0)
    # ^ Select elevation band.
)

# =============================================================================
# PREPARE DEM
# =============================================================================

dem_export = (
    dem_mosaic
    .clip(export_poly)
    # ^ Restrict raster to AOI boundary.

    .reproject(crs=Export_CRS, scale=EXPORT_SCALE)
    # ^ Force correct resolution + projection before export.

    .rename('elevation_m')
    # ^ Assign clear band name.
)

ready_to_export = (
    dem_export
    .toFloat()
    # ^ Ensure Float32 output.

    .unmask(NODATA_VALUE)
    # ^ Fill NoData areas with standard value.
)

# =============================================================================
# INTERACTIVE MAP 
# =============================================================================

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Center map on AOI.

Map.add_basemap('SATELLITE')
# ^ Basemap for visual context.

Map.addLayer(
    dem_mosaic.clip(export_poly),
    {
        'min': 0,
        'max': 500,
        'palette': [
            '006633',
            'E5FFCC',
            'FFCC00',
            '662A00',
            'FFFFFF'
        ]
    },
    'DEM'
)
# ^ DEM visualization.

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ AOI overlay.

display(Map)

# =============================================================================
# EXPORT 
# =============================================================================

task = ee.batch.Export.image.toDrive(
    image=ready_to_export,
    # ^ Final DEM image ready for export.

    description=FILE_PREFIX,
    # ^ Task name in GEE.

    folder=FOLDER,
    # ^ Drive destination.

    fileNamePrefix=FILE_PREFIX,
    # ^ Output filename.

    region=dissolved,
    # ^ ✅ Correct geometry (NOT FeatureCollection)

    scale=EXPORT_SCALE,
    # ^ Output resolution.

    crs=Export_CRS,
    # ^ Output projection.

    maxPixels=int(1e13)
    # ^ Prevents export pixel limit errors.
)

task.start()

# =============================================================================
# STATUS
# =============================================================================

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('DEM EXPORT STARTED')
print('AOI-based export (no partitioning)')
print('NoData value:', NODATA_VALUE)
print('CRS:', Export_CRS)
print('Scale:', EXPORT_SCALE, 'm')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

In [ ]:
# Slope 



# =============================================================================
# 1m SLOPE EXTRACTION (PERCENT)
# =============================================================================

import ee
import geemap

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining study corridor.

Export_CRS = 'EPSG:5070'
# ^ Output coordinate system projection 

FOLDER = 'GEE_Exports_Slope'
# ^ Google Drive export folder.

FILE_PREFIX = 'MN_I-90_SLOPE_PCT'
# ^ Base filename for slope outputs (percent slope).

EXPORT_SCALE = 1
# ^ Output resolution in meters 

NODATA_VALUE = -9999
# ^ GIS-friendly NoData value.

# =============================================================================
# LOAD DATA
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine assets.

dissolved = export_poly.union().geometry()
# ^ Dissolves AOI into a single geometry for consistent processing.

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/10m_collection')
    .filterBounds(export_poly)
    .mosaic()
    .select(0)
)
# ^ Builds seamless 1m DEM from USGS 3DEP dataset.

# =============================================================================
# SLOPE (PERCENT)
# =============================================================================

slope_deg = ee.Terrain.slope(
    dem_mosaic.reproject(crs=Export_CRS, scale=EXPORT_SCALE)
)
# ^ Computes slope in degrees from DEM using local 3x3 neighborhood gradient.
#   Output is angular slope (0–90 degrees).

slope_pct = (
    slope_deg
    .multiply(math.pi / 180)
    # ^ Convert degrees → radians

    .tan()
    # ^ tan(theta) = rise/run (dimensionless slope ratio)

    .multiply(100)
    # ^ Convert ratio → percent slope

    .rename('slope_pct')
)
# ^ Final output is percent slope (0–100+ depending on terrain steepness).

ready_to_export = (
    slope_pct
    .clip(export_poly)
    # ^ Restricts raster to AOI boundary.

    .toFloat()
    # ^ Ensures 32-bit float output.

    .unmask(NODATA_VALUE)
    # ^ Fills masked pixels with GIS NoData value.
)

# =============================================================================
# INTERACTIVE MAP 
# =============================================================================

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Centers map on AOI at regional scale.

Map.add_basemap('SATELLITE')
# ^ Adds imagery basemap for terrain interpretation.

Map.addLayer(
    slope_pct.clip(export_poly),
    {
        'min': 0,
        'max': 100,
        # ^ Visualization range (0% flat → steep terrain >100%)

        'palette': [
            '006633',  # flat / low slope
            'E5FFCC',  # gentle slopes
            'FFCC00',  # moderate slopes
            '662A00',  # steep slopes
            'FFFFFF'   # extreme slope highlights
        ]
    },
    'Slope (%)'
)
# ^ Visual slope layer for QA/QC.

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ AOI boundary overlay.

display(Map)
# ^ Renders interactive map in notebook environment.

# =============================================================================
# EXPORT 
# =============================================================================

task = ee.batch.Export.image.toDrive(
    image=ready_to_export,
    # ^ Final processed slope raster (percent slope).

    description=FILE_PREFIX,
    # ^ Task name in Earth Engine Task Manager.

    folder=FOLDER,
    # ^ Google Drive output folder.

    fileNamePrefix=FILE_PREFIX,
    # ^ Output filename base.

    region=dissolved,
    # ^ Uses true AOI geometry (NOT bounding box) for clean export.

    scale=EXPORT_SCALE,
    # ^ Output resolution 

    crs=Export_CRS,
    # ^ Output coordinate reference system.

    maxPixels=int(1e13)
    # ^ Prevents export failure for large rasters.
)

task.start()
# ^ Submits export task asynchronously.

# =============================================================================
# FINAL STATUS
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✓ SLOPE EXPORT STARTED")
print("AOI-based export (no bounding box)")
print("CRS:", Export_CRS)
print("Scale:", EXPORT_SCALE, "m")
print("Output: Percent slope")
print("Monitor: https://code.earthengine.google.com/tasks")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

In [ ]:
# =============================================================================
# MOSAIC RASTERS — ArcGIS Pro (NoData = -9999)
# =============================================================================
# Run in ArcGIS Pro Python Notebook

import arcpy
import os

# ----------------------------------------
# 1️⃣ INPUTS
# ----------------------------------------

shard_dir  = r"C:\Users\KyleSteen.AzureAD\Downloads\slope_Mosaic"
output_gdb = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\Jessie_Atlanta_CHM\WashDOT.gdb"
output_name = "Mosaic_1"
Export_CRS = 5070

NODATA_VALUE = -9999

# ----------------------------------------
# 2️⃣ COLLECT RASTERS
# ----------------------------------------

shards = [
    os.path.join(shard_dir, f)
    for f in os.listdir(shard_dir)
    if f.lower().endswith(".tif")
]

if not shards:
    raise FileNotFoundError(f"No .tif files found in: {shard_dir}")

shards = sorted(shards)

print(f"Found {len(shards)} raster(s)")

# ----------------------------------------
# 3️⃣ FORCE NODATA STANDARDIZATION
# ----------------------------------------
# This step actually WRITES nodata into the raster

print("Standardizing NoData = -9999...")

clean_shards = []

for shard in shards:
    out_raster = shard.replace(".tif", "_nd.tif")

    arcpy.management.CopyRaster(
        in_raster=shard,
        out_rasterdataset=out_raster,
        nodata_value=str(NODATA_VALUE),
        pixel_type="32_BIT_FLOAT"
    )

    clean_shards.append(out_raster)

print("NoData standardization complete.")

# ----------------------------------------
# 4️⃣ MOSAIC TO NEW RASTER
# ----------------------------------------

input_rasters = ";".join(clean_shards)

print("Mosaicking rasters...")

with arcpy.EnvManager(parallelProcessingFactor="80%"):
    arcpy.management.MosaicToNewRaster(
        input_rasters=input_rasters,
        output_location=output_gdb,
        raster_dataset_name_with_extension=output_name,
        coordinate_system_for_the_raster=arcpy.SpatialReference(Export_CRS),
        pixel_type="32_BIT_FLOAT",
        number_of_bands=1,
        mosaic_method="FIRST",
        mosaic_colormap_mode="FIRST"
    )

print(f"\nComplete: {os.path.join(output_gdb, output_name)}")